In [ ]:
!pip install -q transformers datasets librosa soundfile pdfplumber python-docx joblib

In [ ]:
import os

BASE = "/content/AI-Interview-Resume-Coach"
for folder in [
    f"{BASE}/data/raw",
    f"{BASE}/data/processed",
    f"{BASE}/models/resume_classifier",
    f"{BASE}/models/resume_scorer",
    f"{BASE}/models/sentiment_analyzer",
    f"{BASE}/models/speech_emotion",
]:
    os.makedirs(folder, exist_ok=True)

print("✅ Folders created")

In [ ]:
# OPTION B: Synthetic fallback (run ONLY if you skipped Option A)
import pandas as pd
import random

csv_path = f"{BASE}/data/raw/UpdatedResumeDataSet.csv"

if not os.path.exists(csv_path):
    categories = {
        "Data Science": ["machine learning", "deep learning", "python", "pandas", "numpy",
                          "tensorflow", "pytorch", "data visualization", "statistics", "sql"],
        "Java Developer": ["java", "spring boot", "hibernate", "microservices", "rest api",
                            "maven", "junit", "multithreading", "sql", "design patterns"],
        "Web Designing": ["html", "css", "javascript", "react", "figma", "ui/ux",
                           "responsive design", "bootstrap", "wordpress", "photoshop"],
        "HR": ["recruitment", "onboarding", "payroll", "employee relations",
               "performance management", "hr policies", "talent acquisition", "compliance"],
        "Mechanical Engineer": ["autocad", "solidworks", "thermodynamics", "manufacturing",
                                  "cad design", "fluid mechanics", "project management", "quality control"],
        "Business Analyst": ["requirements gathering", "sql", "tableau", "power bi",
                              "stakeholder management", "process modeling", "agile", "data analysis"],
    }
    rows = []
    for category, skills in categories.items():
        for i in range(40):
            sampled = random.sample(skills, k=min(6, len(skills)))
            text = (f"Experienced professional with strong skills in {', '.join(sampled)}. "
                    f"Worked on multiple projects involving {sampled[0]} and {sampled[-1]}. "
                    f"Seeking opportunities in {category} domain. "
                    f"Education: Bachelor's degree. Years of experience: {random.randint(1, 8)}.")
            rows.append({"Category": category, "Resume": text})
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print("✅ Synthetic resume dataset generated")
else:
    print("✅ Real dataset already present, skipping synthetic generation")

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[\r\n\t]", " ", text)
    text = re.sub(r"[^a-z0-9\s,.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_skills(text, skill_list):
    text_lower = text.lower()
    return [s for s in skill_list if s.lower() in text_lower]

COMMON_SKILLS = [
    "python", "java", "javascript", "c++", "sql", "machine learning",
    "deep learning", "tensorflow", "pytorch", "nlp", "computer vision",
    "data analysis", "data visualization", "pandas", "numpy", "scikit-learn",
    "react", "node.js", "html", "css", "aws", "docker", "kubernetes",
    "git", "rest api", "agile", "communication", "leadership", "teamwork",
    "problem solving", "project management", "tableau", "power bi", "excel"
]

print("✅ Utilities loaded")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv(f"{BASE}/data/raw/UpdatedResumeDataSet.csv")
df = df.dropna(subset=["Resume", "Category"])
df["clean_resume"] = df["Resume"].apply(clean_text)
df = df[df["clean_resume"].str.len() > 10]

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Category"])

print(f"Categories ({len(label_encoder.classes_)}):")
for i, c in enumerate(label_encoder.classes_):
    print(f"  {i}: {c}")

joblib.dump(label_encoder, f"{BASE}/models/label_encoder.joblib")

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

class ResumeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding="max_length",
                              max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

def evaluate(model, loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_preds = torch.argmax(outputs.logits, dim=1)
            preds.extend(batch_preds.cpu().numpy())
            true.extend(labels.cpu().numpy())
    return accuracy_score(true, preds), f1_score(true, preds, average="weighted")

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(label_encoder.classes_)
).to(DEVICE)

train_ds = ResumeDataset(train_df["clean_resume"].tolist(), train_df["label"].tolist(), tokenizer)
val_ds = ResumeDataset(val_df["clean_resume"].tolist(), val_df["label"].tolist(), tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

OUTPUT_DIR = f"{BASE}/models/resume_classifier"
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    acc, f1 = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: val_acc={acc:.4f}, val_f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✅ Best model saved (f1={f1:.4f})")

print(f"\nTraining complete. Best F1: {best_f1:.4f}")

In [ ]:
def predict_category(text, top_k=3):
    cleaned = clean_text(text)
    enc = tokenizer(cleaned, truncation=True, padding="max_length", max_length=256, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).squeeze(0)
    top_probs, top_idx = torch.topk(probs, k=min(top_k, len(label_encoder.classes_)))
    results = []
    for prob, idx in zip(top_probs, top_idx):
        category = label_encoder.inverse_transform([idx.item()])[0]
        results.append({"category": category, "confidence": round(prob.item(), 4)})
    return results

sample = "Experienced in python, machine learning, tensorflow, pytorch, data visualization and sql."
for r in predict_category(sample):
    print(f"{r['category']}: {r['confidence']*100:.2f}%")

In [ ]:
import numpy as np

ACTION_VERBS = ["developed", "designed", "implemented", "managed", "led", "created",
                 "built", "improved", "increased", "reduced", "achieved", "launched",
                 "optimized", "analyzed", "collaborated", "delivered", "automated"]

def extract_features(resume_text):
    text = resume_text or ""
    cleaned = clean_text(text)
    words = cleaned.split()
    word_count = len(words)
    skills_found = extract_skills(cleaned, COMMON_SKILLS)
    num_skills = len(skills_found)
    has_email = 1 if re.search(r"\S+@\S+", text) else 0
    has_phone = 1 if re.search(r"(\+?\d[\d\-\s]{8,}\d)", text) else 0
    num_bullets = text.count("\n") + text.count("•") + text.count("- ")
    sentences = [s for s in re.split(r"[.!?]", cleaned) if len(s.strip()) > 0]
    avg_sentence_len = sum(len(s.split()) for s in sentences) / len(sentences) if sentences else 0
    num_action_verbs = sum(cleaned.count(v) for v in ACTION_VERBS)
    education_mentioned = 1 if any(kw in cleaned for kw in ["bachelor","master","phd","degree","university","college"]) else 0
    exp_match = re.findall(r"(\d+)\s*\+?\s*years?", cleaned)
    experience_years = max([int(x) for x in exp_match], default=0)
    return {
        "word_count": word_count, "num_skills_matched": num_skills,
        "has_email": has_email, "has_phone": has_phone,
        "num_bullets": num_bullets, "avg_sentence_len": round(avg_sentence_len, 2),
        "num_action_verbs": num_action_verbs, "education_mentioned": education_mentioned,
        "experience_years": experience_years, "skills_found": skills_found,
    }

import random as rnd
def synthetic_quality_score(features):
    score = 0
    score += min(features["word_count"] / 5, 25)
    score += min(features["num_skills_matched"] * 3, 24)
    score += features["has_email"] * 5
    score += features["has_phone"] * 5
    score += min(features["num_bullets"] * 1.5, 10)
    score += min(features["num_action_verbs"] * 2, 16)
    score += features["education_mentioned"] * 10
    score += min(features["experience_years"] * 1.5, 10)
    score += rnd.uniform(-3, 3)
    return round(max(0, min(100, score)), 2)

rows = []
for _, row in df.iterrows():
    feats = extract_features(row["Resume"])
    feats_no_skills = {k: v for k, v in feats.items() if k != "skills_found"}
    feats_no_skills["quality_score"] = synthetic_quality_score(feats)
    rows.append(feats_no_skills)

feat_df = pd.DataFrame(rows)
feat_df.to_csv(f"{BASE}/data/processed/resume_features.csv", index=False)
print(f"✅ Feature dataset: {len(feat_df)} rows")
feat_df.head()

In [ ]:
import torch.nn as nn
from torch.utils.data import random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

FEATURE_COLS = ["word_count", "num_skills_matched", "has_email", "has_phone",
                "num_bullets", "avg_sentence_len", "num_action_verbs",
                "education_mentioned", "experience_years"]
TARGET_COL = "quality_score"

class ResumeScoreDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class ResumeScoreMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x)

X = feat_df[FEATURE_COLS].values.astype(np.float32)
y = feat_df[TARGET_COL].values.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, f"{BASE}/models/resume_scorer/scaler.joblib")

dataset = ResumeScoreDataset(X_scaled, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

score_model = ResumeScoreMLP(input_dim=len(FEATURE_COLS)).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(score_model.parameters(), lr=1e-3)

best_mae = float("inf")
for epoch in range(100):
    score_model.train()
    train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds = score_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    score_model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            preds = score_model(xb).cpu().numpy()
            all_preds.extend(preds.flatten())
            all_true.extend(yb.numpy().flatten())

    mae = mean_absolute_error(all_true, all_preds)
    r2 = r2_score(all_true, all_preds)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/100 | val_MAE={mae:.3f} | val_R2={r2:.3f}")

    if mae < best_mae:
        best_mae = mae
        torch.save(score_model.state_dict(), f"{BASE}/models/resume_scorer/model.pt")

print(f"\n✅ Training complete. Best val MAE: {best_mae:.3f}")

In [ ]:
def predict_score(resume_text):
    features = extract_features(resume_text)
    X = np.array([[features[c] for c in FEATURE_COLS]], dtype=np.float32)
    X_scaled = scaler.transform(X)
    with torch.no_grad():
        score = score_model(torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)).item()
    return round(max(0, min(100, score)), 2), features

sample_resume = ("Experienced data scientist with python, machine learning, tensorflow, pytorch, sql skills. "
                  "Email: test@test.com Phone: 9876543210. Developed and implemented several ML models, "
                  "improved accuracy by 20%. Bachelor degree in computer science. 5 years experience.")

score, feats = predict_score(sample_resume)
print(f"Predicted Quality Score: {score}/100")
print(f"Skills found: {feats['skills_found']}")

In [ ]:
CONFIDENT_TEMPLATES = [
    "I successfully led the project and delivered it two weeks ahead of schedule.",
    "I am confident in my ability to solve this problem using {skill}.",
    "I have extensive experience with {skill} and have used it in production for years.",
    "I took full ownership of the migration and it went smoothly with zero downtime.",
    "I am certain this approach will work because I have implemented it before.",
    "My team and I improved system performance by 40% within two months.",
    "I clearly understand the requirements and can implement this efficiently.",
    "I have a proven track record of delivering high-quality results under pressure.",
]
NEUTRAL_TEMPLATES = [
    "I worked on a project that involved {skill}, and it was completed on time.",
    "I have some experience with {skill} from my previous role.",
    "I think this approach could work, depending on the constraints.",
    "I collaborated with the team to finish the {skill} module.",
    "The task was completed, though there were a few minor challenges.",
    "I followed the standard process for handling this kind of issue.",
    "I can give it a try and see how it goes.",
]
HESITANT_TEMPLATES = [
    "I'm not really sure, but maybe I could try using {skill}?",
    "I haven't worked much with {skill}, so I might struggle a bit.",
    "I think... I'm not totally confident about this answer.",
    "Um, I guess it could work, but I'm not 100% sure.",
    "I'm a bit nervous about this topic, I don't have much experience.",
    "I'm not certain, I would probably need to look it up.",
    "Sorry, I don't fully remember how that works.",
]
SKILLS = ["python", "machine learning", "react", "sql", "system design", "data structures",
          "cloud deployment", "testing", "agile methodology", "api design"]

rnd.seed(42)
conf_rows = []
for label, templates in [("confident", CONFIDENT_TEMPLATES), ("neutral", NEUTRAL_TEMPLATES), ("hesitant", HESITANT_TEMPLATES)]:
    for _ in range(150):
        template = rnd.choice(templates)
        skill = rnd.choice(SKILLS)
        text = template.format(skill=skill) if "{skill}" in template else template
        conf_rows.append({"text": text, "label": label})

conf_df = pd.DataFrame(conf_rows).sample(frac=1, random_state=42).reset_index(drop=True)
conf_df.to_csv(f"{BASE}/data/processed/interview_confidence_dataset.csv", index=False)
print(f"✅ Generated {len(conf_df)} samples")
print(conf_df["label"].value_counts())

In [ ]:
from collections import Counter

MAX_LEN = 30
EMBED_DIM = 64
HIDDEN_DIM = 64

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens[:max_len]]
    ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

class ConfidenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.X = [encode(t, vocab) for t in texts]
        self.y = labels
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

class ConfidenceBiLSTM(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, num_classes),
        )
    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h)

conf_label_encoder = LabelEncoder()
conf_df["label_id"] = conf_label_encoder.fit_transform(conf_df["label"])
joblib.dump(conf_label_encoder, f"{BASE}/models/sentiment_analyzer/label_encoder.joblib")

vocab = build_vocab(conf_df["text"].tolist())
joblib.dump(vocab, f"{BASE}/models/sentiment_analyzer/vocab.joblib")

conf_dataset = ConfidenceDataset(conf_df["text"].tolist(), conf_df["label_id"].tolist(), vocab)
train_size = int(0.8 * len(conf_dataset))
val_size = len(conf_dataset) - train_size
conf_train, conf_val = random_split(conf_dataset, [train_size, val_size])

conf_train_loader = DataLoader(conf_train, batch_size=16, shuffle=True)
conf_val_loader = DataLoader(conf_val, batch_size=16)

conf_model = ConfidenceBiLSTM(vocab_size=len(vocab), num_classes=len(conf_label_encoder.classes_)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(conf_model.parameters(), lr=1e-3)

def eval_conf(model, loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            pred = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(pred)
            true.extend(yb.numpy())
    return accuracy_score(true, preds), f1_score(true, preds, average="weighted")

best_f1 = 0.0
for epoch in range(20):
    conf_model.train()
    total_loss = 0
    for xb, yb in conf_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = conf_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    acc, f1 = eval_conf(conf_model, conf_val_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/20 | loss={total_loss/len(conf_train_loader):.4f} | val_acc={acc:.4f} | val_f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(conf_model.state_dict(), f"{BASE}/models/sentiment_analyzer/model.pt")

print(f"\n✅ Training complete. Best val F1: {best_f1:.4f}")

In [ ]:
def predict_confidence(text):
    ids = encode(text, vocab)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = conf_model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0)
    pred_idx = torch.argmax(probs).item()
    label = conf_label_encoder.inverse_transform([pred_idx])[0]
    return label, round(probs[pred_idx].item(), 4)

label, conf = predict_confidence("I am confident I can solve this using python.")
print(f"Predicted: {label} ({conf*100:.2f}%)")

label, conf = predict_confidence("Um, I'm not really sure, but maybe I could try?")
print(f"Predicted: {label} ({conf*100:.2f}%)")

In [ ]:
import glob
import soundfile as sf

RAVDESS_DIR = f"{BASE}/data/raw/ravdess"
SYNTHETIC_DIR = f"{BASE}/data/raw/synthetic_audio"

EMOTION_MAP = {"01": "neutral", "02": "calm", "03": "happy", "04": "sad",
               "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"}
SIMPLIFIED_MAP = {"neutral": "calm", "calm": "calm", "happy": "confident", "surprised": "confident",
                 "sad": "nervous", "fearful": "nervous", "angry": "stressed", "disgust": "stressed"}

audio_rows = []
ravdess_files = glob.glob(os.path.join(RAVDESS_DIR, "**", "*.wav"), recursive=True)

if ravdess_files:
    print(f"✅ Found {len(ravdess_files)} RAVDESS files")
    for f in ravdess_files:
        name = os.path.basename(f).split(".")[0]
        parts = name.split("-")
        if len(parts) >= 3:
            emotion = EMOTION_MAP.get(parts[2], "unknown")
            simplified = SIMPLIFIED_MAP.get(emotion, "unknown")
            audio_rows.append({"file_path": f, "emotion": emotion, "interview_label": simplified})
else:
    print("⚠️ No RAVDESS found. Generating synthetic audio...")
    os.makedirs(SYNTHETIC_DIR, exist_ok=True)
    emotion_params = {
        "calm": {"freq": 220, "noise": 0.01}, "confident": {"freq": 330, "noise": 0.02},
        "nervous": {"freq": 440, "noise": 0.08}, "stressed": {"freq": 550, "noise": 0.15},
    }
    sr = 16000
    duration = 2.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    for emotion, params in emotion_params.items():
        for i in range(15):
            freq = params["freq"] + np.random.uniform(-10, 10)
            noise = np.random.normal(0, params["noise"], t.shape)
            wave = 0.3 * np.sin(2 * np.pi * freq * t) + noise
            fpath = os.path.join(SYNTHETIC_DIR, f"{emotion}_{i}.wav")
            sf.write(fpath, wave.astype(np.float32), sr)
            audio_rows.append({"file_path": fpath, "emotion": emotion, "interview_label": emotion})

audio_df = pd.DataFrame(audio_rows)
audio_df.to_csv(f"{BASE}/data/processed/speech_emotion_labels.csv", index=False)
print(audio_df["interview_label"].value_counts())

In [ ]:
import librosa

N_MFCC = 40
AUDIO_MAX_LEN = 130

def extract_mfcc(file_path, n_mfcc=N_MFCC, max_len=AUDIO_MAX_LEN):
    y, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

class SpeechEmotionDataset(Dataset):
    def __init__(self, features, labels):
        self.X = torch.tensor(features, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class SpeechEmotionCNN(nn.Module):
    def __init__(self, num_classes, n_mfcc=N_MFCC, max_len=AUDIO_MAX_LEN):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        conv_out_h = n_mfcc // 4
        conv_out_w = max_len // 4
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * conv_out_h * conv_out_w, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

speech_label_encoder = LabelEncoder()
audio_df["label_id"] = speech_label_encoder.fit_transform(audio_df["interview_label"])
joblib.dump(speech_label_encoder, f"{BASE}/models/speech_emotion/label_encoder.joblib")

print("Extracting MFCC features...")
audio_features = np.stack([extract_mfcc(f) for f in tqdm(audio_df["file_path"].tolist())])

mean, std = audio_features.mean(), audio_features.std()
audio_features = (audio_features - mean) / (std + 1e-8)
joblib.dump({"mean": mean, "std": std}, f"{BASE}/models/speech_emotion/norm_stats.joblib")

speech_dataset = SpeechEmotionDataset(audio_features, audio_df["label_id"].values)
train_size = int(0.8 * len(speech_dataset))
val_size = len(speech_dataset) - train_size
speech_train, speech_val = random_split(speech_dataset, [train_size, val_size])

speech_train_loader = DataLoader(speech_train, batch_size=8, shuffle=True)
speech_val_loader = DataLoader(speech_val, batch_size=8)

speech_model = SpeechEmotionCNN(num_classes=len(speech_label_encoder.classes_)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(speech_model.parameters(), lr=1e-3)

best_f1 = 0.0
for epoch in range(30):
    speech_model.train()
    total_loss = 0
    for xb, yb in speech_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = speech_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    acc, f1 = eval_conf(speech_model, speech_val_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/30 | loss={total_loss/len(speech_train_loader):.4f} | val_acc={acc:.4f} | val_f1={f1:.4f}")

    if f1 >= best_f1:
        best_f1 = f1
        torch.save(speech_model.state_dict(), f"{BASE}/models/speech_emotion/model.pt")

print(f"\n✅ Training complete. Best val F1: {best_f1:.4f}")

In [ ]:
def predict_emotion(audio_path):
    mfcc = extract_mfcc(audio_path)
    mfcc = (mfcc - mean) / (std + 1e-8)
    x = torch.tensor(mfcc, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = speech_model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0)
    pred_idx = torch.argmax(probs).item()
    label = speech_label_encoder.inverse_transform([pred_idx])[0]
    return label, round(probs[pred_idx].item(), 4)

test_file = audio_df["file_path"].iloc[0]
label, conf = predict_emotion(test_file)
print(f"File: {test_file}")
print(f"Predicted: {label} ({conf*100:.2f}%)")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/trained_models", "zip", f"{BASE}/models")
files.download("/content/trained_models.zip")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install PyPDF2

In [ ]:
import PyPDF2

pdf_file = list(uploaded.keys())[0]

text = ""

with open(pdf_file, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

print(text[:2000])

In [ ]:
resume_text = clean_text(text)

inputs = tokenizer(
    resume_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

category = label_encoder.inverse_transform([pred])[0]

print("Predicted Category:", category)